In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
#Tokenization Parameters:

num_words: Vocabulary size (e.g., 2000, 5000)

maxlen: Maximum sequence length (e.g., 10, 20, 50)

Embedding Layer:

input_dim: Vocabulary size (e.g., 1000, 5000)

output_dim: Dimensionality of embedding vectors (e.g., 64, 128)

LSTM Layer:

units: Number of units in the LSTM layer (e.g., 64, 128, 256)

return_sequences: Whether to return the full sequence or just the last output (e.g., True or False)

Dense Layer:

units: Number of neurons in the Dense layer (e.g., 32, 64, 128)

activation: Activation function for Dense layers (e.g., 'relu', 'sigmoid')

Optimizer:

learning_rate: Learning rate for the optimizer (e.g., 0.0001, 0.001, 0.005)

optimizer: Type of optimizer (e.g., Adam, AdamW, RMSprop)

Training Parameters:

batch_size: Number of samples per batch (e.g., 16, 32, 64)

epochs: Number of epochs (e.g., 5, 10, 20)

validation_split: Fraction of training data used for validation (e.g., 0.1, 0.2)

Dropout and Regularization (Optional):

Dropout: Dropout rate to prevent overfitting (e.g., 0.2, 0.5)

L2 Regularization: L2 penalty for weights (e.g., 1e-4, 1e-5)

Model Architecture (Optional):

Bidirectional LSTM: Use Bidirectional LSTM (e.g., Bidirectional(LSTM(128)))

SyntaxError: invalid syntax (<ipython-input-4-2571218653>, line 3)

In [ ]:
import json  #hereee

# Load JSON file
with open('/content/drive/MyDrive/Skin Dataset/training split/train.json', 'r') as f:
    json_questions = json.load(f)

# Now `data` is a Python dictionary or list depending on the JSON content
print(json_questions[0]["question"])


What is the name for this disease?


In [ ]:
#text dataset preprocessing  hereeeeeee


import random

questions=[]
answers=[]

added_question = ["Is the disease contagious?",
                  "Is it contagious?",
                  "What is the probability that this is contagious?",
                  "What is the probability that this disease is contagious?",
                  "What are the probability that this is contagious?",
                  "Is it contagious or not?"
                  ]


for i in range(0, len(json_questions), 7):
  questions.append(json_questions[0]["question"])
  questions.append(json_questions[1]["question"])
  questions.append(json_questions[2]["question"])
  questions.append(json_questions[3]["question"])
  questions.append(json_questions[4]["question"])
  questions.append(json_questions[5]["question"])
  questions.append(json_questions[6]["question"])
  questions.append(random.choice(added_question))
  answers.append("name")
  answers.append("cancerous")
  answers.append("features")
  answers.append("severity")
  answers.append("preventions")
  answers.append("test")
  answers.append("cause")
  answers.append("contagious")

In [ ]:
print(f"JSON Question len:{len(json_questions)}\nQuestion len:{len(questions)}\nAnswer len:{len(answers)}")

JSON Question len:5838
Question len:6672
Answer len:6672


In [ ]:
print(len(questions)/7)

953.1428571428571


**LSTM** model

In [ ]:
# 1.1

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Encode labels (answers)
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(answers)

# Tokenize questions
tokenizer = Tokenizer(num_words=3000, oov_token="<OOV>")
tokenizer.fit_on_texts(questions)
sequences = tokenizer.texts_to_sequences(questions)
padded = pad_sequences(sequences, maxlen=20, padding='post')

# Build LSTM model
text_model = Sequential([
    Embedding(input_dim=3000, output_dim=64, input_length=20),
    LSTM(256),
    Dense(64, activation='gelu'),
    Dense(len(set(labels)), activation='softmax')  # Output = number of unique answers
])

text_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#1.2

from tensorflow.keras.optimizers import Adam

# Hyperparameters
optimizer = Adam(learning_rate=0.001)
loss_function = 'sparse_categorical_crossentropy'
metrics = ['accuracy']

# Compile model with selected hyperparameters
text_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# 1.3
# Train model
text_model.fit(padded, np.array(labels), epochs=10, batch_size=16, validation_split=0.2)

# Prediction function
def predict_answer(question):
    seq = tokenizer.texts_to_sequences([question])
    pad = pad_sequences(seq, maxlen=10, padding='post')
    pred = np.argmax(text_model.predict(pad), axis=-1)
    return label_encoder.inverse_transform(pred)[0]

# Test prediction
print("Prediction:", predict_answer("What are the preventive measures for this disease?"))


Epoch 1/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5815 - loss: 0.9451 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 2/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 1.0000 - val_loss: 4.3980e-04
Epoch 3/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 1.0000 - loss: 3.6200e-04 - val_accuracy: 1.0000 - val_loss: 2.0385e-04
Epoch 4/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 1.0000 - loss: 1.7658e-04 - val_accuracy: 1.0000 - val_loss: 1.1625e-04
Epoch 5/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 1.0000 - loss: 1.0375e-04 - val_accuracy: 1.0000 - val_loss: 7.3919e-05
Epoch 6/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 1.0000 - loss: 6.7344e-05 - val_accuracy: 1.0000 - val_loss: 5.0214e-05
Epoch 7/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 1.0000 - loss: 4.6065e-05 - val_accuracy: 1.0000 - val_loss: 3.5694e-05
Epoch 8/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accur

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2.1

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Encode labels (answers)
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(answers)

# Tokenize questions
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(questions)
sequences = tokenizer.texts_to_sequences(questions)
padded = pad_sequences(sequences, maxlen=10, padding='post')

# Build LSTM model
text_model = Sequential([
    Embedding(input_dim=1000, output_dim=64, input_length=20),
    LSTM(256),
    Dense(32, activation='sigmoid'),
    Dense(len(set(labels)), activation='softmax')  # Output = number of unique answers
])

text_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#2.2

from tensorflow.keras.optimizers import Adam

# Hyperparameters
optimizer = Adam(learning_rate=0.01)
loss_function = 'sparse_categorical_crossentropy'
metrics = ['accuracy']

# Compile model with selected hyperparameters
text_model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
#2.3
# Train model
text_model.fit(padded, np.array(labels), epochs=10, batch_size=16, validation_split=0.1)

# Prediction function
def predict_answer(question):
    seq = tokenizer.texts_to_sequences([question])
    pad = pad_sequences(seq, maxlen=10, padding='post')
    pred = np.argmax(text_model.predict(pad), axis=-1)
    return label_encoder.inverse_transform(pred)[0]

# Test prediction
print("Prediction:", predict_answer("What are the preventive measures for this disease?"))



Epoch 1/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.8016 - loss: 0.5964 - val_accuracy: 1.0000 - val_loss: 0.0064
Epoch 2/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 0.0047 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 3/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 9.9700e-04
Epoch 4/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.0000 - loss: 8.7767e-04 - val_accuracy: 1.0000 - val_loss: 5.9940e-04
Epoch 5/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 1.0000 - loss: 5.3986e-04 - val_accuracy: 1.0000 - val_loss: 3.8439e-04
Epoch 6/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 1.0000 - loss: 3.5167e-04 - val_accuracy: 1.0000 - val_loss: 2.6859e-04
Epoch 7/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 1.0000 - loss: 2.4832e-04 - val_accuracy: 1.0000 - val_loss: 1.9595e-04
Epoch 8/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 1.00

In [ ]:
#commentss

#1
Training Accuracy: 100% (1.0000) throughout the epochs.

Validation Accuracy: 100% (1.0000) throughout the epochs.

Training Loss: Starts at 1.8278e-06 and decreases to nearly zero, with loss values like 0, 1e-10, and 3.7971e-10.

Validation Loss: Similarly, it decreases to near zero, indicating almost perfect fitting.



In [ ]:
#2
raining Accuracy: 100% (1.0000) by Epoch 2

Validation Accuracy: 100% (1.0000) throughout the training

Training Loss: Starts at 0.5964 and decreases significantly to 1.0631e-04 by Epoch 10.

Validation Loss: Starts at 0.0064 and decreases to 8.8246e-05 by Epoch 10.

In [ ]:
####

The first model is likely better because it reaches very low loss values almost immediately (indicating better convergence). It seems to generalize well since the loss decreases to near-zero values quickly.

The second model shows slower convergence, with the loss dropping more gradually, though still improving significantly.



In [ ]:
# 3.1

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Encode labels (answers)
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(answers)

# Tokenize questions
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(questions)
sequences = tokenizer.texts_to_sequences(questions)
padded = pad_sequences(sequences, maxlen=10, padding='post')

# Build LSTM model
text_model = Sequential([
    Embedding(input_dim=5000, output_dim=32, input_length=20),
    LSTM(8),
    Dense(8, activation='sigmoid'),
    Dense(len(set(labels)), activation='softmax')  # Output = number of unique answers
])

text_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#3.2

from tensorflow.keras.optimizers import AdamW

# Hyperparameters
optimizer = AdamW(learning_rate=0.01)
loss_function = 'categorical_crossentropy'
metrics = ['accuracy']

# Compile model with selected hyperparameters
text_model.compile(optimizer=optimizer, loss='loss_function', metrics=['accuracy'])


In [ ]:
#3.3
# Train model
text_model.fit(padded, np.array(labels), epochs=15, batch_size=32, validation_split=0.2)

# Prediction function
def predict_answer(question):
    seq = tokenizer.texts_to_sequences([question])
    pad = pad_sequences(seq, maxlen=10, padding='post')
    pred = np.argmax(text_model.predict(pad), axis=-1)
    return label_encoder.inverse_transform(pred)[0]

# Test prediction
print("Prediction:", predict_answer("What are the preventive measures for this disease?"))



Epoch 1/15


ValueError: Could not interpret loss identifier: loss_function